In [1]:
# python version
import sys
print(sys.executable)

C:\Users\felip\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import time

sys.path.append(os.path.abspath('../src'))

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from PipelineBuilder import Pipeline
from CrossValidation import CrossValidation

In [3]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# pytorch threads
n_threads = os.cpu_count()
torch.set_num_threads(n_threads)
n_threads

8

In [5]:
## Loading data
data_loader = DataLoader(path_train="../data/train.csv", path_eval="../data/validation.csv", path_test="../data/test.csv")
train, validation, test = data_loader.load()
train.shape, validation.shape, test.shape

((29000, 2), (1000, 2), (10000, 2))

In [6]:
## Splitting data
data_splitter = DataSplitter(src_language='english', target_language='portuguese')
X_train, y_train = data_splitter.split(train)
X_val, y_val = data_splitter.split(validation)
X_test, y_test = data_splitter.split(test)

## LSTM

In [7]:
# DL Pipeline 
params = {'model_name': "LSTM", 'tokenizer_method': "sentencepiece", 'vocab_size': 3000, 'max_length': 20, 'embedding_dim': 128, 'hidden_dim': 128, 
          'num_layers': 1, 'dropout': 0.3, 'teacher_forcing': 0.5, 'max_length_decoded': 20, 'lr': 1e-3, 'epochs': 1, 'batch_size': 128} 
params['device'] = "gpu"

pipeline = Pipeline(**params)
pipeline

,max_length,20
,vocab_size,3000
,model_name,'LSTM'
,embedding_dim,128
,hidden_dim,128
,num_layers,1
,dropout,0.3
,teacher_forcing,0.5
,max_length_decoded,20
,lr,0.001
,epochs,1


In [8]:
# Cross-Validation
cv = CrossValidation(pipeline=clone(pipeline))

# Holdout Validation
#cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test)
#print("Holdout Validation score: ", cv_score)

In [9]:
## Hyper-parameter tunning
cv = CrossValidation(pipeline=clone(pipeline))

param_grid = {
    'tokenizer_method': ["sentencepiece"],
    'max_length': [30],
    'vocab_size': [5000],
    'max_length_decoded': [30],
    'embedding_dim': [512], 
    'hidden_dim': [512], 
    'num_layers': [2],
    'teacher_forcing': [0.7], 
    'lr': [7e-4], 
    'epochs': [1], 
    'batch_size': [128]
}

#cv_lstm_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
#                                                               reduced=True, train_size=10000, return_loss_curves=True)
#cv_lstm_results

In [10]:
#cv.pipeline.set_params(epochs=150)
#cv.pipeline.fit(X_train, y_train)
#cv.pipeline.save("translatorLSTM2.pt")

In [11]:
#cv.pipeline.score(X_test, y_test)

In [12]:
#preds = cv.pipeline.predict(X_test)
#preds_df = pd.DataFrame({'english': X_test.iloc[:, 0].values, 'portuguese': y_test.iloc[:, 0], 'translation': preds})
#preds_df.tail(50)

## Transformer (encoder-decoder)

In [13]:
# DL Pipeline 
params = {'model_name': "Transformer", 'tokenizer_method': "full words", 'vocab_size': 3000, 'max_length': 30, 'embedding_dim': 256, 
          'n_heads': 8, 'num_encoder_layers': 2, 'num_decoder_layers': 2, 'dim_feed_forward': 512, 'dropout': 0.1, 'max_length_decoded': 30, 
          'lr': 1e-3, 'epochs': 1, 'batch_size': 256} 
params['device'] = "gpu"

pipeline = Pipeline(**params)
pipeline

,tokenizer_method,'full words'
,max_length,30
,vocab_size,3000
,embedding_dim,256
,n_heads,8
,dim_feed_forward,512
,dropout,0.1
,max_length_decoded,30
,lr,0.001
,epochs,1
,batch_size,256


In [14]:
# Cross-Validation
cv = CrossValidation(pipeline=clone(pipeline))

# Holdout Validation
#cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test)
#print("Holdout Validation score: ", cv_score)

In [15]:
## Hyper-parameter tunning
cv = CrossValidation(pipeline=clone(pipeline))

param_grid = {
    'tokenizer_method': ["full words"],
    'max_length': [30],
    'max_length_decoded': [30],
    'vocab_size': [4000],
    'embedding_dim': [256],  
    'n_heads': [8],
    'num_encoder_layers': [4],
    'num_decoder_layers': [4],
    'dim_feed_forward': [1024],
    'dropout': [0.1],
    'lr': [5e-4], 
    'epochs': [1], 
    'batch_size': [128]
}

cv_trans_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
                                                               reduced=True, train_size=10000, return_loss_curves=True)
cv_trans_results

Time cleaning:  1.585247278213501
vocab size: 2452.
Full Words
7.0669
[ 7. 10. 11. 15.]
vocab size: 3114.
Full Words
7.0661
[ 7. 10. 12. 15.]
Time tokenizing:  0.4105653762817383
Training Activated ? True
Loss after Epoch 1: 4.7306.
forward time: 12.997188200126402
backward time:  21.77096840098966
loss time:  0.39616900018882006
Loss after gradient descent: 4.730642.
Total Training time: 41.42674398422241.
Params:  {'batch_size': 128, 'dim_feed_forward': 1024, 'dropout': 0.1, 'embedding_dim': 256, 'epochs': 1, 'lr': 0.0005, 'max_length': 30, 'max_length_decoded': 30, 'n_heads': 8, 'num_decoder_layers': 4, 'num_encoder_layers': 4, 'tokenizer_method': 'full words', 'vocab_size': 4000}
Fit time: 43.75220251083374
Full Words
7.0843
[ 7. 10. 11. 15.]
time cleaning and encoding test:  0.9500818252563477
Training Activated ? False
time predicting test:  126.8571367263794
Time decodig test:  0.0352778434753418
Prediction time: 127.84249639511108

Total CV time: 173.67884612083435



,batch_size,dim_feed_forward,dropout,embedding_dim,epochs,lr,max_length,max_length_decoded,n_heads,num_decoder_layers,num_encoder_layers,tokenizer_method,vocab_size,fit_time,pred_time,final_loss,final_validation_score,score
0,128,1024,0.1,256,1,0.0005,30,30,8,4,4,full words,4000,43.752203,127.842496,4.730642,NaN,1.434746


In [16]:
#cv.pipeline.set_params(epochs=120)
#cv.pipeline.fit(X_train, y_train)
#cv.pipeline.save("translatorTransformer1.pt")

In [17]:
#cv.pipeline.score(X_test, y_test)

In [18]:
#preds = cv.pipeline.predict(X_test)
#preds_df = pd.DataFrame({'english': X_test.iloc[:, 0].values, 'portuguese': y_test.iloc[:, 0], 'translation': preds})
#preds_df.tail(50)

In [19]:
cv.pipeline.set_params(epochs=120, tokenizer_method="sentencepiece")
cv.pipeline.fit(X_train, y_train)
cv.pipeline.save("translatorTransformer2.pt")

Time cleaning:  5.214784145355225
SentencePiece
8.294206896551724
[ 8. 12. 13. 18.]
SentencePiece
8.151862068965517
[ 8. 12. 14. 18.]
Time tokenizing:  6.788647174835205
Training Activated ? True
Loss after Epoch 1: 4.6756.
Loss after Epoch 6: 1.7443.
Loss after Epoch 11: 0.9702.
Loss after Epoch 16: 0.6119.
Loss after Epoch 21: 0.4370.
Loss after Epoch 26: 0.3495.
Loss after Epoch 31: 0.2976.
Loss after Epoch 36: 0.2616.
Loss after Epoch 41: 0.2374.
Loss after Epoch 46: 0.2198.
Loss after Epoch 51: 0.2059.
Loss after Epoch 56: 0.1921.
Loss after Epoch 61: 0.1817.
Loss after Epoch 66: 0.1716.
Loss after Epoch 71: 0.1635.
Loss after Epoch 76: 0.1562.
Loss after Epoch 81: 0.1501.
Loss after Epoch 86: 0.1453.
Loss after Epoch 91: 0.1405.
Loss after Epoch 96: 0.1367.
Loss after Epoch 101: 0.1337.
Loss after Epoch 106: 0.1255.
Loss after Epoch 111: 0.1235.
Loss after Epoch 116: 0.1215.
forward time: 5074.172743302188
backward time:  9543.968869599514
loss time:  415.7167689919006
Loss after

In [20]:
cv.pipeline.score(X_test, y_test)

SentencePiece
8.3897
[ 8. 12. 14. 18.]
time cleaning and encoding test:  7.341774225234985
Training Activated ? False
time predicting test:  110.85435819625854
Time decodig test:  0.7064669132232666


35.935175450718326

In [21]:
preds = cv.pipeline.predict(X_test)
preds_df = pd.DataFrame({'english': X_test.iloc[:, 0].values, 'portuguese': y_test.iloc[:, 0], 'translation': preds})
preds_df.tail(50)

SentencePiece
8.3897
[ 8. 12. 14. 18.]
time cleaning and encoding test:  2.9923479557037354
Training Activated ? False
time predicting test:  93.90228319168091
Time decodig test:  0.6447384357452393


,english,portuguese,translation
9950,I have a lot of homework.,Eu tenho muito dever de casa.,Eu tenho muita lição de casa.
9951,This has to be stopped.,Isso tem que ser barrado.,Isso deve estar.
9952,I do hope so.,Eu realmente espero.,Assim espero.
9953,Tom doesn't like Mary very much.,Tom não gosta muito de Maria.,O Tom não gosta muito de a Mary.
9954,Are you worried about global warming?,Você está preocupado com o aquecimento global?,Você está preocupado com sono ou não?
9955,Tom is big and strong.,Tom é grande e forte.,Tom é grande e forte.
9956,Tom can draw a perfect circle.,Tom sabe desenhar um círculo perfeito.,Tom pode desenhar um crculo.
9957,These children are my grandchildren.,Estas crianças são meus netos.,Estes são meus filhos eletas.
9958,I don't like that part of town.,Não gosto daquela parte da cidade.,Eu não gosto da cidade.
9959,He led a life of luxury.,Ele levou uma vida de luxo.,Ele vçaria uma vida na vida.


In [22]:
preds_df.head(60)

,english,portuguese,translation
0,Everything matters.,Tudo importa.,Tudo está importa.
1,I'm going to get something to eat.,Vou comer qualquer coisa.,Vou pegar alguma coisa para comer.
2,Isn't there a pharmacy nearby?,Não há uma farmácia por aqui?,A próxima vez eu já não tem uma farsa?
3,The secret of longevity is to choose your pare...,Escolher os seus pais cuidadosamente é o segre...,A torre vai escolher sua falta com cuidado.
4,Tom isn't very likely to forget to do that.,Tom provavelmente não vai esquecer de fazer isso.,O Tom provavelmente não é muito provável que f...
5,They're both empty.,Ambos estão vazios.,Eles são vazio.
6,"We waited a long time, but Tom never came.",Nós esperamos por muito tempo mas o Tom nunca ...,"Nunca vimos tempo Tom, mas nunca apareceu."
7,They held the meeting here.,Eles fizeram a reunião aqui.,Eles prender a reunião aqui.
8,Are you ready to leave?,Você está pronto para partir?,Você está pronto para partir?
9,I heard Tom partied all night.,Ouvi Tom festejando a noite toda.,Eu ouvi Tom festejando a noite toda.
